# B02: Hai model và cách chấm

Dùng model pretrained, không training. Notebook chưa có kết quả cho đến khi các cell chạy thật.

Chọn GPU trong Runtime. Khi được hỏi, upload `stt_b01_b03_package.zip`. Dữ liệu: 10 FLEURS train pilot và 100 FLEURS dev cố định, seed 42. Không chạy test.

B03 có thể chạy cả ba buổi từ đầu; các output vẫn được tách B01/B02/B03. Không so thời gian Zipformer CPU với Whisper/Qwen GPU như cùng thiết bị.

In [ ]:
from pathlib import Path
import sys, zipfile, subprocess
ROOT = Path('/content/PBL6')
if not (ROOT / 'experiments/common.py').exists():
    # Upload stt_b01_b03_package.zip from the computer; no Drive login required.
    from google.colab import files
    uploaded = files.upload()
    package = next(name for name in uploaded if name.endswith('.zip'))
    with zipfile.ZipFile(package) as archive:
        for name in archive.namelist():
            target = (ROOT / name).resolve()
            if not target.is_relative_to(ROOT.resolve()):
                raise ValueError('Unsafe ZIP path')
        archive.extractall(ROOT)
sys.path.insert(0, str(ROOT / 'experiments'))
from common import read_csv, read_jsonl, score
def run(*args):
    subprocess.run([sys.executable, *map(str, args)], cwd=ROOT, check=True)
print('Workspace:', ROOT)


In [ ]:
run('-m', 'pip', 'install', 'openai-whisper==20250625', 'soundfile==0.14.0', 'jiwer==4.0.0', 'sherpa-onnx==1.13.8')
run('-m', 'unittest', 'discover', '-s', ROOT / 'experiments/tests', '-v')


In [ ]:
import torch, platform
print('Python:', platform.python_version())
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if not torch.cuda.is_available():
    raise RuntimeError('Chọn Runtime > Change runtime type > GPU rồi chạy lại.')


## B01: chạy baseline

Audio là đầu vào; reference chỉ để chấm. Mỗi lần chạy lưu prediction và trạng thái từng mẫu. Thời gian nạp model được ghi riêng.

Pi hiện có: Pi 4B 8GB, OS 13 Trixie aarch64. Micro/màn hình: chưa xác nhận.

In [ ]:
run(ROOT / 'experiments/run_asr.py', '--model', 'whisper',
    '--manifest', ROOT / 'experiments/manifests/pilot.csv',
    '--output', ROOT / 'experiments/B01/results/whisper_pilot.jsonl', '--device', 'cuda')


In [ ]:
from IPython.display import Audio, display
rows = read_jsonl(ROOT / 'experiments/B01/results/whisper_pilot.jsonl')
r = rows[0]
display(Audio(filename=str(ROOT / r['audio_path'])))
print('Reference:', r['reference'])
print('Prediction:', r['prediction'])
print('Scorer:', score(r['reference'], r['prediction']))
print('Ví dụ tính tay (minh họa): tôi đi học → tôi học: D=1, N=3, WER_ws=1/3')


## B02: Whisper và Zipformer

WER_ws tách bằng khoảng trắng, gần mức âm tiết tiếng Việt. CER đếm Unicode code point sau NFC, bỏ khoảng trắng. Cả reference và prediction dùng cùng chuẩn hóa, giữ dấu và chữ số. Điểm toàn tập là tổng lỗi/tổng độ dài nhãn. Inference lỗi giữ status và tính dự đoán rỗng, không loại mẫu.

In [ ]:
for model in ['whisper', 'zipformer']:
    run(ROOT / 'experiments/run_asr.py', '--model', model,
        '--manifest', ROOT / 'experiments/manifests/fleurs_dev100.csv',
        '--output', ROOT / f'experiments/B02/results/{model}_dev.jsonl',
        '--device', 'cuda' if model == 'whisper' else 'cpu')
run(ROOT / 'experiments/summarize.py',
    '--manifest', ROOT / 'experiments/manifests/fleurs_dev100.csv',
    '--runs', ROOT / 'experiments/B02/results/whisper_dev.jsonl', ROOT / 'experiments/B02/results/zipformer_dev.jsonl',
    '--output-dir', ROOT / 'experiments/B02/results')


In [ ]:
import pandas as pd
display(pd.read_csv(ROOT / 'experiments/B02/results/summary.csv'))
display(pd.read_csv(ROOT / 'experiments/B02/results/comparison.csv').head(3))


## Lưu kết quả

Chạy cell này trước khi ngắt runtime. Dữ liệu đã lưu là bằng chứng; notebook có mã nhưng chưa chạy không phải kết quả. Không dùng điểm Colab để kết luận tốc độ trên Pi.

In [ ]:
import shutil
from google.colab import files
# Contains real predictions, environment and summaries; excludes model weights.
export = Path('/content/PBL6_results_B01_B03')
export.mkdir(exist_ok=True)
for stage in ['B01', 'B02', 'B03']:
    source = ROOT / f'experiments/{stage}/results'
    if source.exists():
        shutil.copytree(source, export / stage, dirs_exist_ok=True)
shutil.copytree(ROOT / 'experiments/manifests', export / 'manifests', dirs_exist_ok=True)
archive = shutil.make_archive(str(export), 'zip', export)
files.download(archive)
print('Lưu ZIP vào G:/My Drive/Seventh Semester/PBL6/STT_B01_B03 sau khi tải.')
